# Lekcja 11: Pandas I — Rozwiązania zadań

Rozwiązania wszystkich 20 zadań z lekcji o wczytywaniu i czyszczeniu danych. Każde zadanie poprzedzone pełną treścią.

> Kod napisany pod **Pandas 3.0+**: wypełnianie braków zawsze przez przypisanie (`df['kol'] = df['kol'].fillna(...)`), `ffill`/`bfill` jako metody (nie `method=`).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.core.interchange.dataframe_protocol import DataFrame
%matplotlib inline

URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic_raw = pd.read_csv(URL)          # wczytujemy raz, w zadaniach robimy .copy()
print('pandas', pd.__version__, '| Titanic:', titanic_raw.shape)

pandas 3.0.3 | Titanic: (891, 12)


## Zadania podstawowe (1-8)

### Zadanie 1 — Tworzenie DataFrame z danych

Stwórz DataFrame zawierający informacje o 10 produktach w sklepie: nazwa, cena, ilość na stanie, kategoria.

**Wymagania:**
- Użyj słownika do utworzenia DataFrame
- Wyświetl pierwsze 5 wierszy
- Oblicz całkowitą wartość towaru (cena × ilość) i dodaj jako nową kolumnę

**Oczekiwany rezultat:** DataFrame z 10 produktami i kolumną `WartośćCałkowita`.

*(proste)*

### Zadanie 2 — Wczytanie CSV i podstawowe statystyki

Wczytaj dataset Titanic i oblicz podstawowe statystyki.

Dataset: `https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv`

**Wymagania:**
- Wczytaj dane przez `pd.read_csv()`
- Wyświetl `.info()` i `.describe()`
- Oblicz średni wiek pasażerów

*(proste)*

### Zadanie 3 — Selekcja kolumn

Z datasetu Titanic wybierz tylko kolumny: Name, Age, Sex, Survived.

**Wymagania:**
- Stwórz nowy DataFrame z wybranymi kolumnami
- Wyświetl pierwsze 10 wierszy
- Zapisz do pliku `titanic_subset.csv`

*(proste)*

In [3]:
titanic_raw.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [48]:
titanic_select = titanic_raw[['Name', 'Age', 'Sex', 'Survived']]

print(titanic_select.head(10))

titanic_select.to_csv('titanic_subset.csv', index=False)

                                                Name   Age     Sex  Survived
0                            Braund, Mr. Owen Harris  22.0    male         0
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0  female         1
2                             Heikkinen, Miss. Laina  26.0  female         1
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0  female         1
4                           Allen, Mr. William Henry  35.0    male         0
5                                   Moran, Mr. James   NaN    male         0
6                            McCarthy, Mr. Timothy J  54.0    male         0
7                     Palsson, Master. Gosta Leonard   2.0    male         0
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  27.0  female         1
9                Nasser, Mrs. Nicholas (Adele Achem)  14.0  female         1


### Zadanie 4 — Filtrowanie danych

Znajdź wszystkich pasażerów Titanica, którzy mieli mniej niż 18 lat i przeżyli katastrofę.

**Wymagania:**
- Użyj boolean indexing
- Policz ile było takich osób
- Wyświetl ich imiona i wiek

*(proste)*

In [54]:
onboard_below_18_survived = titanic_raw[(titanic_raw['Age'] < 18) & (titanic_raw['Survived'] == 1)]

print(f"Przeżyło: {len(onboard_below_18_survived)}")

print(onboard_below_18_survived[['Name', 'Age']])

Przeżyło: 61
                                         Name    Age
9         Nasser, Mrs. Nicholas (Adele Achem)  14.00
10            Sandstrom, Miss. Marguerite Rut   4.00
22                McGowan, Miss. Anna "Annie"  15.00
39                Nicola-Yarred, Miss. Jamila  14.00
43   Laroche, Miss. Simonne Marie Anne Andree   3.00
..                                        ...    ...
830   Yasbeck, Mrs. Antoni (Selini Alexander)  15.00
831           Richards, Master. George Sibley   0.83
853                 Lines, Miss. Mary Conover  16.00
869           Johnson, Master. Harold Theodor   4.00
875          Najib, Miss. Adele Kiamie "Jane"  15.00

[61 rows x 2 columns]


### Zadanie 5 — Wykrywanie NaN

Sprawdź które kolumny w Titanicu mają brakujące dane.

**Wymagania:**
- Policz NaN w każdej kolumnie
- Oblicz procent NaN
- Wyświetl tylko kolumny z >10% braków

*(proste)*

In [64]:
nan_count = titanic_raw.isna().sum()
nan_percent = (nan_count / len(titanic_raw)) * 100

nan_summary = pd.DataFrame({'NaN_count': nan_count, 'NaN_percent': nan_percent})
print(nan_summary)

             NaN_count  NaN_percent
PassengerId          0     0.000000
Survived             0     0.000000
Pclass               0     0.000000
Name                 0     0.000000
Sex                  0     0.000000
Age                177    19.865320
SibSp                0     0.000000
Parch                0     0.000000
Ticket               0     0.000000
Fare                 0     0.000000
Cabin              687    77.104377
Embarked             2     0.224467


### Zadanie 6 — Wypełnianie NaN średnią

Wypełnij brakujące wartości w kolumnie `Age` średnią.

**Wymagania:**
- Oblicz średnią wieku (bez NaN)
- Wypełnij NaN tą średnią
- Sprawdź czy nie ma już NaN w `Age`

*(proste)*

### Zadanie 7 — Sortowanie DataFrame

Posortuj pasażerów Titanica po wieku (od najmłodszego do najstarszego).

**Wymagania:**
- Użyj `.sort_values()`
- Wyświetl 10 najmłodszych
- Wyświetl 10 najstarszych

*(proste)*

### Zadanie 8 — Filtrowanie przez `.query()`

Użyj metody `.query()` aby znaleźć pasażerów 1. klasy, którzy zapłacili więcej niż 50£.

**Wymagania:**
- Użyj `.query()` z warunkiem złożonym
- Policz ile było takich pasażerów
- Oblicz średnią cenę ich biletów

*(proste)*

## Zadania średnie (9-12)

### Zadanie 9 — Batch processing

Wczytaj dataset Titanic w batchach po 100 wierszy i oblicz średni wiek w każdym batchu.

**Wymagania:**
- Użyj `chunksize=100`
- Dla każdego batcha wypisz numer i średni wiek
- Oblicz globalną średnią ze wszystkich batchy

*(średnie)*

### Zadanie 10 — Kompleksowe czyszczenie danych

Wyczyść dataset Titanic stosując pełny pipeline: usuń kolumny z >60% NaN, wypełnij Age medianą, wypełnij Embarked dominantą, usuń wiersze z pozostałymi NaN.

**Wymagania:**
- Przed i po: wypisz kształt DataFrame
- Sprawdź czy nie ma NaN
- Porównaj rozkład Age przed i po

*(średnie)*

### Zadanie 11 — Feature engineering

Stwórz nowe zmienne z Titanic: `FamilySize` = SibSp + Parch + 1; `IsAlone` = 1 jeśli FamilySize == 1, inaczej 0; `AgeGroup` = kategorie (0-12, 13-18, 19-35, 36-60, 60+).

**Wymagania:**
- Dodaj wszystkie 3 zmienne do DataFrame
- Wyświetl `value_counts()` dla każdej
- Oblicz wskaźnik przeżycia dla IsAlone vs nie-alone

*(średnie)*

In [67]:
titanic_raw

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [66]:
titanic_feature = titanic_raw.copy()
titanic_feature['FamilySize'] = titanic_feature['SibSp'] + titanic_feature['Parch'] + 1
titanic_feature['IsAlone'] = np.where(titanic_feature['FamilySize'] == 1, 1, 0)

print(titanic_feature['FamilySize'].value_counts())
print(titanic_feature['IsAlone'].value_counts())

alone_surivival_rate = titanic_feature.groupby('IsAlone')['Survived'].mean()

print(f"Survival rate dla osób samotnych (IsAlone=1): {alone_surivival_rate[1]:.2f}")
print(f"Survival rate dla osób nie samotnych (IsAlone=0): {alone_surivival_rate[0]:.2f}")

FamilySize
1     537
2     161
3     102
4      29
6      22
5      15
7      12
11      7
8       6
Name: count, dtype: int64
IsAlone
1    537
0    354
Name: count, dtype: int64
Survival rate dla osób samotnych (IsAlone=1): 0.30
Survival rate dla osób nie samotnych (IsAlone=0): 0.51


### Zadanie 12 — Optymalizacja typów danych

Zoptymalizuj typy danych w Titanicu aby zużywać mniej pamięci.

**Wymagania:**
- Zmień Pclass, Survived, SibSp, Parch na int8
- Zmień Sex, Embarked na category
- Przed i po: wypisz zużycie pamięci (`.memory_usage(deep=True)`)
- Oblicz % redukcji

*(średnie)*

## Zadania wyzwanie (13-20)

### Zadanie 13 — Interpolacja szeregów czasowych

Stwórz szereg czasowy z lukami i wypełnij go różnymi metodami.

**Wymagania:**
- Stwórz DataFrame z datami (20 dni) i wartościami (niektóre NaN)
- Wypełnij przez: ffill, bfill, interpolację liniową
- Zwizualizuj wszystkie 4 wersje na jednym wykresie
- Porównaj wyniki

*(challenge)*

### Zadanie 14 — Analiza missing data patterns

Przeanalizuj wzorce brakujących danych w Titanicu.

**Wymagania:**
- Stwórz heatmap brakujących danych
- Sprawdź czy NaN w Age koreluje z NaN w innych kolumnach
- Oblicz wskaźnik przeżycia dla wierszy z NaN vs bez NaN w Age

*(challenge)*

### Zadanie 15 — Custom aggregation podczas batch loading

Wczytaj Titanic w batchach i oblicz agregacje bez ładowania całości.

**Wymagania:**
- `chunksize=50`
- Oblicz: total passengers, average age, survival rate — bez konkatenacji wszystkich batchy
- Użyj zmiennych akumulacyjnych
- Porównaj wynik z ładowaniem pełnego DataFrame

*(challenge)*

### Zadanie 16 — Zaawansowane filtrowanie

Znajdź rodziny (>1 osoby), gdzie przeżyła połowa lub więcej członków.

**Wymagania:**
- Stwórz zmienną FamilyID (kombinacja nazwiska + FamilySize)
- Grupuj po FamilyID, oblicz survival rate
- Filtruj rodziny z survival >= 50%
- Wyświetl top 10 "najszczęśliwszych rodzin"

*(challenge)*

In [70]:
titanic_feature['FamilyID'] = titanic_feature['Name'].str.split(',').str[0] + '_' + titanic_feature['FamilySize'].astype(str)
(titanic_feature.groupby('FamilyID')['Survived']
    .mean()
    .reset_index(name='SurvivalRate')
    .query('SurvivalRate >= 0.5')
    .sort_values(by='SurvivalRate', ascending=False)
    .head(10))



,FamilyID,SurvivalRate
696,de Mulder_1,1.0
695,de Messemaeker_2,1.0
688,Young_1,1.0
6,Aks_2,1.0
7,Albimona_1,1.0
685,Woolner_1,1.0
638,Toomey_1,1.0
632,Thorne_1,1.0
631,Thomas_2,1.0
628,Taylor_2,1.0


### Zadanie 17 — Korelacja między brakami danych

Zbadaj czy braki w różnych kolumnach są skorelowane.

**Wymagania:**
- Stwórz DataFrame z boolean (True=NaN, False=not NaN) dla każdej kolumny
- Oblicz macierz korelacji między tymi kolumnami
- Zwizualizuj jako heatmap
- Zinterpretuj: czy NaN w jednej kolumnie zwiększa szansę NaN w innej?

*(challenge)*

### Zadanie 18 — Smart data cleaning pipeline

Stwórz funkcję `clean_titanic()` która automatycznie czyści dane.

**Wymagania:**
- Input: surowy Titanic DataFrame
- Output: czysty DataFrame
- Pipeline: drop high-NaN cols, fill Age/Embarked, optimize dtypes, add FamilySize
- Funkcja powinna działać na dowolnym podobnym datasecie

*(challenge)*

### Zadanie 19 — Outlier detection i obsługa

Znajdź i obsłuż outliers w kolumnie Fare.

**Wymagania:**
- Wykryj outliers metodą IQR (Q1-1.5×IQR, Q3+1.5×IQR)
- Zwizualizuj przed i po (boxplot)
- Zastosuj 3 strategie: usunięcie, cap (przycinanie), transformacja log
- Porównaj efekty

*(challenge)*

### Zadanie 20 — Efektywność pamięciowa — benchmark

Porównaj zużycie pamięci różnych metod wczytywania.

**Wymagania:**
- Zmierz pamięć dla: pełne wczytanie, usecols (5 kolumn), chunksize, optymalizacja dtypes
- Użyj `.memory_usage(deep=True)` i psutil
- Stwórz wykres słupkowy porównujący zużycie
- Oblicz oszczędności energii (założenie: 1 MB RAM = 0.0003 kWh/h)

*(challenge)*